# Big Data Analysis of Google Play Store Applications / Group 22
###### Team Members: Lotta Kauppinen, Jenna Kiviaho, Marjaana Koski, Jani Laakso, Aleksi Savukoski

#### The chosen dataset contains information of more than 2.3 million applications from Google Play Store. The main goal of this analysis is to find different patterns between the variables. By doing that, we are able to analyze how the different characteristics affect the popularity and ratings.

#### The analysis also aims to showcase if the free apps outperform the paid apps.

#### The other goal for this analysis is to showcase how Spark and MongoDB work together in a Big Data Analysis.


In [4]:
#  Spark Session:

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("GooglePlayStoreAnalysis").getOrCreate()

In [5]:
# Downloading the dataset:
df = spark.read.csv("data/Google-Playstore.csv",header=True,inferSchema=True)
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating: string (nullable = true)
 |-- Rating Count: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: string (nullable = true)
 |-- Maximum Installs: string (nullable = true)
 |-- Free: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Currency: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Developer Website: string (nullable = true)
 |-- Developer Email: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Privacy Policy: string (nullable = true)
 |-- Ad Supported: string (nullable = true)
 |-- In App Purchases: string (nullable = true)
 |-- Editors Choice: string (nullable = true)
 |-- Scra

In [6]:
# The amount of rows and columns we are working with:
print("Rows:", df.count())
print("Columns:", len(df.columns))

Rows: 2312944
Columns: 24


### Cleaning the data

In [7]:
# Checking the amount of null values by column:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|App Name|App Id|Category|Rating|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Currency|Size|Minimum Android|Developer Id|Developer Website|Developer Email|Released|Last Updated|Content Rating|Privacy Policy|Ad Supported|In App Purchases|Editors Choice|Scraped Time|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+--------+----+---------------+------------+-----------------+---------------+--------+------------+--------------+--------------+------------+----------------+--------------+------------+
|       0|     0|       0| 22883|       22883|     107|             107|               0|   0|    0|     135| 196|           6530|      

#### Dropping columns that are not needed

In [8]:
# Due to the purpose of this analysis, we are not going to need the information of the columns: 
# Currency (USD), Developer Website, Developer Email, Privacy Policy, Scraped Time, so let's drop them.
df = df.drop("Currency","Developer Website","Developer Email","Privacy Policy","Scraped Time")

#### Handling NULL-values

In [9]:
# Let's check the remaining nulls and make a plan to handle them.
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+
|App Name|App Id|Category|Rating|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Size|Minimum Android|Developer Id|Released|Last Updated|Content Rating|Ad Supported|In App Purchases|Editors Choice|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+
|       0|     0|       0| 22883|       22883|     107|             107|               0|   0|    0| 196|           6530|          33|   71055|           0|             0|           4|               2|             0|
+--------+------+--------+------+------------+--------+----------------+----------------+----+-----+----+---------------+-----------

#### Rating and Rating Count -columns: NULL-values
##### These columns includes the average rating and number of rating of each app. All of the NULL-values and values that are not numeric will be changed to 0.0 (Rating) and 0 (Rating Count).

In [10]:
from pyspark.sql.functions import col

# Let's find the rows that include values that ar NOT int or float.
non_numeric = df.filter(~col("Rating").rlike(r'^-?\d+(\.\d+)?$'))

# Examples of the values that are NOT int or float.
non_numeric.select("Rating").show(100, truncate=False)


+--------------------------------+
|Rating                          |
+--------------------------------+
|Books & Reference               |
|com.andromo.dev807396.app1066982|
|net.jp.apps.hirotsuchiya.kanji  |
|Education                       |
|com.storyshare.wisdom           |
|Shopping                        |
|Productivity                    |
|com.app.aziuyr                  |
|Educational                     |
|Social                          |
|com.menshchyna.android          |
|kr.manse.catdog                 |
|com.heliconia.estudio           |
|Education                       |
|Productivity                    |
|Shopping                        |
|Productivity                    |
|Food & Drink                    |
|Racing                          |
|com.ksmps.cricketalpha          |
|Business                        |
|Entertainment                   |
|News & Magazines                |
|Board                           |
|Finance                         |
|Entertainment      

In [11]:
from pyspark.sql.functions import col, when, replace

# It seems that some of the App-values have been written in the wrong columns. Let's fill all the non-numeric values with 0.0.
df = df.withColumn(
    "Rating_num",
    when(col("Rating").rlike(r'^-?\d+(\.\d+)?$'), col("Rating").cast("float"))
    .otherwise(0.0)
)

# To check that the code worked:
df.select("Rating", "Rating_num").filter(col("Rating_num") == 0.0).show(100, truncate=False)

+------+----------+
|Rating|Rating_num|
+------+----------+
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |
|0.0   |0.0       |


In [12]:
df = df.withColumn(
    "Rating Count", 
    when(col("Rating_num") == 0.0, "0")  # Rating_num == 0.0 > Rating Count = 0
    .otherwise(col("Rating Count"))      
)

# To check that the code worked:
df.select("Rating", "Rating_num", "Rating Count").filter(col("Rating_num") == 0.0).show(100, truncate=False)
df = df.drop("Rating")

+------+----------+------------+
|Rating|Rating_num|Rating Count|
+------+----------+------------+
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.0       |0           |
|0.0   |0.

### Changing the data types

In [13]:
# Checking the data types
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating Count: string (nullable = true)
 |-- Installs: string (nullable = true)
 |-- Minimum Installs: string (nullable = true)
 |-- Maximum Installs: string (nullable = true)
 |-- Free: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Size: string (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Last Updated: string (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Ad Supported: string (nullable = true)
 |-- In App Purchases: string (nullable = true)
 |-- Editors Choice: string (nullable = true)
 |-- Rating_num: double (nullable = true)



In [14]:
from pyspark.sql.functions import translate, when, round, expr, lower

# Changing data types
# Using try_cast instead of cast for unexpected values
df = df.withColumn("Installs", expr("try_cast(translate(Installs, '+,', '') as long)")) # Removing characters '+' and ','
df = df.withColumn("Price", expr("try_cast(Price as float)"))
df = df.withColumn("Rating Count", expr("try_cast(`Rating Count` as int)"))
df = df.withColumn("Free", 
    when(lower(col("Free").cast("string")) == "true", True)
    .when(lower(col("Free").cast("string")) == "false", False)
    .otherwise(None)
)

# Handling column "Size"
df = df.withColumn("Size", 
    when(col("Size").contains("k"),
         expr("try_cast(translate(Size, 'k,', ' .') as float)") / 1024) # Removing 'k' and ',', then divided by 1024 so every size is in MB
    .when(col("Size").contains("M"), 
         expr("try_cast(translate(Size, 'M,', ' .') as float)")) # Removing characters 'M' and ','
    .otherwise(expr("try_cast(Size as float)")) # If it's only a number and no characters
)

# Rounding column "Size"
df = df.withColumn("Size", round(col("Size"), 2))

# Checking data types again to make sure they are correct
print(df.schema["Installs"].dataType)
print(df.schema["Price"].dataType)
print(df.schema["Rating Count"].dataType)
print(df.schema["Free"].dataType)
print(df.schema["Size"].dataType)


LongType()
FloatType()
IntegerType()
BooleanType()
DoubleType()


In [15]:
df = df.withColumn("Ad Supported", 
    when(lower(col("Ad Supported").cast("string")) == "true", True)
    .when(lower(col("Ad Supported").cast("string")) == "false", False)
    .otherwise(None)
)
df = df.withColumn("Editors Choice", 
    when(lower(col("Editors Choice").cast("string")) == "true", True)
    .when(lower(col("Editors Choice").cast("string")) == "false", False)
    .otherwise(None)
)
df = df.withColumn("In App Purchases", 
    when(lower(col("In App Purchases").cast("string")) == "true", True)
    .when(lower(col("In App Purchases").cast("string")) == "false", False)
    .otherwise(None)
)

df = df.withColumn("Minimum Installs", expr("try_cast(`Minimum Installs` as long)"))
df = df.withColumn("Maximum Installs", expr("try_cast(`Maximum Installs` as long)"))

print(df.schema["Ad Supported"].dataType)
print(df.schema["Editors Choice"].dataType)
print(df.schema["In App Purchases"].dataType)
print(df.schema["Minimum Installs"].dataType)
print(df.schema["Maximum Installs"].dataType)

BooleanType()
BooleanType()
BooleanType()
LongType()
LongType()


#### Dropping columns with null values

#### We retained Size and Released columns despite their ~6% null rate. Removing them would narrow the data significantly and could have potentially biased the analysis of other attributes like popularity and categories.

In [16]:
df = df.na.drop(subset=['Installs', 'Minimum Installs', 'Ad Supported', 'Developer Id', 'In App Purchases', 'Free', 'Minimum Android'])
df.count()

2306230

#### Handling Duplicates
##### Due to the nature of this data, we need to check only the rows that have the exact same values in each column. Because there are no such rows, nothing needs to be deleted.

In [17]:
from pyspark.sql.functions import count

df.groupBy(df.columns).count().filter("count > 1").show()

+--------+------+--------+------------+--------+----------------+----------------+----+-----+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+----------+-----+
|App Name|App Id|Category|Rating Count|Installs|Minimum Installs|Maximum Installs|Free|Price|Size|Minimum Android|Developer Id|Released|Last Updated|Content Rating|Ad Supported|In App Purchases|Editors Choice|Rating_num|count|
+--------+------+--------+------------+--------+----------------+----------------+----+-----+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+----------+-----+
+--------+------+--------+------------+--------+----------------+----------------+----+-----+----+---------------+------------+--------+------------+--------------+------------+----------------+--------------+----------+-----+



### Handling Released and Last Updated columns

In [18]:
df.select("Released", "Last Updated").show(10)
print(df.schema["Released"].dataType)
print(df.schema["Last Updated"].dataType)

+------------+------------+
|    Released|Last Updated|
+------------+------------+
|Feb 26, 2020|Feb 26, 2020|
|May 21, 2020|May 06, 2021|
| Aug 9, 2019|Aug 19, 2019|
|Sep 10, 2018|Oct 13, 2018|
|Feb 21, 2020|Nov 12, 2018|
|Dec 24, 2018|Dec 20, 2019|
|Sep 23, 2019|Sep 27, 2019|
|Jun 21, 2019|Jun 21, 2019|
|        NULL|Dec 07, 2018|
|Sep 22, 2019|Oct 07, 2020|
+------------+------------+
only showing top 10 rows
StringType()
StringType()


#### We will change the dates into DateType because they are now String. The original format doesn't fit for the analysis, but DateType allows us to perform time-based analysis, such as grouping app releases by month or year

In [19]:
from pyspark.sql.functions import try_to_date

# Using try_to_date function so unexpected values will be filled with NULL
df = df.withColumn("Released", try_to_date(col("Released"), "MMM d, yyyy")) \
       .withColumn("Last Updated", try_to_date(col("Last Updated"), "MMM d, yyyy"))

# Checking the outcome
df.select("Released", "Last Updated").show(20)
print(df.schema["Released"].dataType)
print(df.schema["Last Updated"].dataType)


+----------+------------+
|  Released|Last Updated|
+----------+------------+
|2020-02-26|  2020-02-26|
|2020-05-21|  2021-05-06|
|2019-08-09|  2019-08-19|
|2018-09-10|  2018-10-13|
|2020-02-21|  2018-11-12|
|2018-12-24|  2019-12-20|
|2019-09-23|  2019-09-27|
|2019-06-21|  2019-06-21|
|      NULL|  2018-12-07|
|2019-09-22|  2020-10-07|
|2020-07-30|  2020-07-30|
|2018-01-10|  2018-06-27|
|2018-04-03|  2021-06-11|
|2020-02-09|  2021-05-14|
|2018-09-05|  2020-05-30|
|2020-04-05|  2021-03-23|
|2016-11-28|  2019-10-30|
|2019-04-24|  2019-05-05|
|2020-07-01|  2021-05-26|
|2020-12-26|  2021-03-23|
+----------+------------+
only showing top 20 rows
DateType()
DateType()


In [20]:
df.printSchema()

root
 |-- App Name: string (nullable = true)
 |-- App Id: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Rating Count: integer (nullable = true)
 |-- Installs: long (nullable = true)
 |-- Minimum Installs: long (nullable = true)
 |-- Maximum Installs: long (nullable = true)
 |-- Free: boolean (nullable = true)
 |-- Price: float (nullable = true)
 |-- Size: double (nullable = true)
 |-- Minimum Android: string (nullable = true)
 |-- Developer Id: string (nullable = true)
 |-- Released: date (nullable = true)
 |-- Last Updated: date (nullable = true)
 |-- Content Rating: string (nullable = true)
 |-- Ad Supported: boolean (nullable = true)
 |-- In App Purchases: boolean (nullable = true)
 |-- Editors Choice: boolean (nullable = true)
 |-- Rating_num: double (nullable = true)



### Analysing the dependency of app ratings and price type.

In [21]:
from pyspark.sql.functions import col, avg, stddev, count, when, round as spark_round
import pandas as pd
import matplotlib.pyplot as plt

# Register temp view
df.createOrReplaceTempView("apps")

# ============================================================
# ANALYSIS 1: BASIC STATISTICS - FREE VS PAID RATINGS
# ============================================================

print("="*60)
print("RATING STATISTICS: FREE VS PAID APPS")
print("="*60)

rating_stats = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        COUNT(*) AS total_apps,
        COUNT(CASE WHEN Rating_num IS NOT NULL AND Rating_num != 'nan' THEN 1 END) AS apps_with_ratings,
        ROUND(AVG(CASE WHEN Rating_num != 'nan' THEN CAST(Rating_num AS FLOAT) ELSE NULL END), 2) AS avg_rating,
        ROUND(STDDEV(CASE WHEN Rating_num != 'nan' THEN CAST(Rating_num AS FLOAT) ELSE NULL END), 2) AS stddev_rating,
        ROUND(MIN(CASE WHEN Rating_num != 'nan' THEN CAST(Rating_num AS FLOAT) ELSE NULL END), 2) AS min_rating,
        ROUND(MAX(CASE WHEN Rating_num != 'nan' THEN CAST(Rating_num AS FLOAT) ELSE NULL END), 2) AS max_rating,
        ROUND(PERCENTILE_APPROX(CASE WHEN Rating_num != 'nan' THEN CAST(Rating_num AS FLOAT) ELSE NULL END, 0.5), 2) AS median_rating
    FROM apps
    WHERE Rating_num IS NOT NULL AND Rating_num != 'nan'
    GROUP BY Free
""")

rating_stats.show(truncate=False)

# ============================================================
# ANALYSIS 2: RATING DISTRIBUTION BY PRICE TYPE
# ============================================================

print("\n" + "="*60)
print("RATING DISTRIBUTION: FREE VS PAID")
print("="*60)

rating_distribution = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        CASE 
            WHEN CAST(Rating_num AS FLOAT) >= 4.5 THEN '4.5 - 5.0 (Excellent)'
            WHEN CAST(Rating_num AS FLOAT) >= 4.0 THEN '4.0 - 4.5 (Good)'
            WHEN CAST(Rating_num AS FLOAT) >= 3.5 THEN '3.5 - 4.0 (Average)'
            WHEN CAST(Rating_num AS FLOAT) >= 3.0 THEN '3.0 - 3.5 (Below Average)'
            WHEN CAST(Rating_num AS FLOAT) >= 2.0 THEN '2.0 - 3.0 (Poor)'
            ELSE 'Below 2.0 (Very Poor)'
        END AS rating_category,
        COUNT(*) AS app_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY Free), 2) AS percentage
    FROM apps
    WHERE Rating_num IS NOT NULL AND Rating_num != 'nan' AND CAST(Rating_num AS FLOAT) IS NOT NULL
    GROUP BY Free, rating_category
    ORDER BY Free DESC, rating_category
""")

rating_distribution.show(20, truncate=False)

# ============================================================
# ANALYSIS 3: STATISTICAL SIGNIFICANCE (T-TEST APPROXIMATION)
# ============================================================

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TEST")
print("="*60)

significance = spark.sql("""
    WITH stats AS (
        SELECT 
            Free,
            COUNT(*) AS n,
            AVG(CAST(Rating_num AS FLOAT)) AS mean,
            STDDEV(CAST(Rating_num AS FLOAT)) AS stddev,
            VARIANCE(CAST(Rating_num AS FLOAT)) AS variance
        FROM apps
        WHERE Rating_num IS NOT NULL AND Rating_num != 'nan' AND CAST(Rating_num AS FLOAT) IS NOT NULL
        GROUP BY Free
    )
    SELECT 
        MAX(CASE WHEN Free = true THEN mean END) AS free_mean,
        MAX(CASE WHEN Free = false THEN mean END) AS paid_mean,
        MAX(CASE WHEN Free = true THEN stddev END) AS free_stddev,
        MAX(CASE WHEN Free = false THEN stddev END) AS paid_stddev,
        MAX(CASE WHEN Free = true THEN n END) AS free_n,
        MAX(CASE WHEN Free = false THEN n END) AS paid_n,
        ABS(MAX(CASE WHEN Free = true THEN mean END) - MAX(CASE WHEN Free = false THEN mean END)) AS mean_difference
    FROM stats
""").collect()[0]

print(f"Free apps average rating: {significance['free_mean']:.2f}")
print(f"Paid apps average rating: {significance['paid_mean']:.2f}")
print(f"Difference: {significance['mean_difference']:.2f}")

# Calculate effect size (Cohen's d approximation)
pooled_std = ((significance['free_stddev']**2 + significance['paid_stddev']**2) / 2) ** 0.5
effect_size = significance['mean_difference'] / pooled_std

print(f"\nEffect size (Cohen's d): {effect_size:.3f}")
if abs(effect_size) < 0.2:
    print("Interpretation: Negligible effect")
elif abs(effect_size) < 0.5:
    print("Interpretation: Small effect")
elif abs(effect_size) < 0.8:
    print("Interpretation: Medium effect")
else:
    print("Interpretation: Large effect")

# ============================================================
# ANALYSIS 4: TOP RATED APPS BY PRICE TYPE
# ============================================================

print("\n" + "="*60)
print("TOP 5 HIGHEST RATED FREE APPS")
print("="*60)

top_free = spark.sql("""
    SELECT 
        `App Name` AS app_name,
        Category,
        ROUND(CAST(Rating_num AS FLOAT), 2) AS rating,
        `Rating Count` AS review_count
    FROM apps
    WHERE Free = true 
        AND Rating_num IS NOT NULL 
        AND Rating_num != 'nan'
        AND CAST(Rating_num AS FLOAT) IS NOT NULL
    ORDER BY CAST(Rating_num AS FLOAT) DESC
    LIMIT 5
""")
top_free.show(truncate=False)

print("\n" + "="*60)
print("TOP 5 HIGHEST RATED PAID APPS")
print("="*60)

top_paid = spark.sql("""
    SELECT 
        `App Name` AS app_name,
        Category,
        ROUND(CAST(Rating_num AS FLOAT), 2) AS rating,
        `Rating Count` AS review_count,
        CONCAT('$', Price) AS price
    FROM apps
    WHERE Free = false 
        AND Rating_num IS NOT NULL 
        AND Rating_num != 'nan'
        AND CAST(Rating_num AS FLOAT) IS NOT NULL
    ORDER BY CAST(Rating_num AS FLOAT) DESC
    LIMIT 5
""")
top_paid.show(truncate=False)

# ============================================================
# ANALYSIS 5: SUMMARY AND CONCLUSION (SKIPPING CORRELATION)
# ============================================================

print("\n" + "="*60)
print("SUMMARY AND CONCLUSIONS")
print("="*60)

summary = spark.sql("""
    SELECT 
        CASE WHEN Free = true THEN 'Free' ELSE 'Paid' END AS price_type,
        COUNT(*) AS total_apps,
        ROUND(AVG(CAST(Rating_num AS FLOAT)), 2) AS avg_rating,
        ROUND(PERCENTILE_APPROX(CAST(Rating_num AS FLOAT), 0.5), 2) AS median_rating,
        ROUND(STDDEV(CAST(Rating_num AS FLOAT)), 3) AS rating_stddev,
        COUNT(CASE WHEN CAST(Rating_num AS FLOAT) >= 4.0 THEN 1 END) AS high_rated_apps,
        ROUND(COUNT(CASE WHEN CAST(Rating_num AS FLOAT) >= 4.0 THEN 1 END) * 100.0 / COUNT(*), 1) AS pct_high_rated
    FROM apps
    WHERE Rating_num IS NOT NULL AND Rating_num != 'nan' AND CAST(Rating_num AS FLOAT) IS NOT NULL
    GROUP BY Free
""")

summary.show(truncate=False)

# Statistical conclusion
free_avg = summary.filter(col("price_type") == "Free").select("avg_rating").collect()[0][0]
paid_avg = summary.filter(col("price_type") == "Paid").select("avg_rating").collect()[0][0]

print(f"\n📊 KEY FINDINGS:")
print(f"• Free apps average rating: {free_avg:.2f}")
print(f"• Paid apps average rating: {paid_avg:.2f}")
print(f"• Difference: {abs(free_avg - paid_avg):.2f} points")
print(f"• Effect size (Cohen's d): {effect_size:.3f}")

# Correct interpretation based on effect size
if effect_size < 0.2:
    print("\n• CONCLUSION: NO meaningful dependency between rating and price type")
    print("  The rating difference is negligible - less than 0.2 standard deviations")
elif effect_size < 0.5:
    print("\n• CONCLUSION: Weak dependency - Paid apps tend to have slightly higher ratings")
elif effect_size < 0.8:
    print("\n• CONCLUSION: Moderate dependency - Paid apps have noticeably higher ratings")
else:
    print("\n• CONCLUSION: Strong dependency - Paid apps have substantially higher ratings")


RATING STATISTICS: FREE VS PAID APPS
+----------+----------+-----------------+----------+-------------+----------+----------+-------------+
|price_type|total_apps|apps_with_ratings|avg_rating|stddev_rating|min_rating|max_rating|median_rating|
+----------+----------+-----------------+----------+-------------+----------+----------+-------------+
|Free      |2261548   |2261548          |2.18      |2.11         |0.0       |5.0       |2.8          |
|Paid      |44682     |44682            |2.36      |2.11         |0.0       |5.0       |3.4          |
+----------+----------+-----------------+----------+-------------+----------+----------+-------------+


RATING DISTRIBUTION: FREE VS PAID
+----------+-------------------------+---------+----------+
|price_type|rating_category          |app_count|percentage|
+----------+-------------------------+---------+----------+
|Free      |2.0 - 3.0 (Poor)         |71696    |3.17      |
|Free      |3.0 - 3.5 (Below Average)|103776   |4.59      |
|Free    